# Form candidate metrics for response curves

This notebook refactors the current response-curve workflow into reusable functions, runs the requested target analyses, and saves fitted curves, metrics, logs, and figures under `explore/no0/form_metrics_260519/`. It is designed to run from the project root.


## 1. Setup and imports

Import the numerical, fitting, and plotting dependencies. Optional fitting libraries are detected and logged instead of being required.


In [1]:
from pathlib import Path
import os
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    LOWESS_AVAILABLE = True
    LOWESS_IMPORT_ERROR = ""
except Exception as exc:
    LOWESS_AVAILABLE = False
    LOWESS_IMPORT_ERROR = repr(exc)

try:
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    SKLEARN_AVAILABLE = True
    SKLEARN_IMPORT_ERROR = ""
except Exception as exc:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = repr(exc)

try:
    from pygam import LinearGAM, s
    PYGAM_AVAILABLE = True
    PYGAM_IMPORT_ERROR = ""
except Exception as exc:
    PYGAM_AVAILABLE = False
    PYGAM_IMPORT_ERROR = repr(exc)

try:
    from gplearn.genetic import SymbolicRegressor
    GPLEARN_AVAILABLE = True
    GPLEARN_IMPORT_ERROR = ""
except Exception as exc:
    GPLEARN_AVAILABLE = False
    GPLEARN_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "0.35",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "0.82",
    "grid.linestyle": "-",
    "grid.linewidth": 0.8,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "savefig.dpi": 300,
})

print("Optional method availability:")
print({
    "LOWESS": LOWESS_AVAILABLE,
    "Polynomial/sklearn": SKLEARN_AVAILABLE,
    "GAM/pygam": PYGAM_AVAILABLE,
    "Symbolic/gplearn": GPLEARN_AVAILABLE,
})


Matplotlib created a temporary cache directory at /var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/matplotlib-fg1r2qoj because the default path (/Users/mimi/.matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.
Optional method availability:
{'LOWESS': True, 'Polynomial/sklearn': True, 'GAM/pygam': True, 'Symbolic/gplearn': True}


## 2. Configuration

Define analysis settings, output folders, method names, plotting style, metric thresholds, and requested targets.


In [2]:
# The notebook should be run from the project root. If it is launched from a
# subfolder, walk upward to the GSA-work root so relative paths still resolve.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "GSA-work":
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if candidate.name == "GSA-work":
            PROJECT_ROOT = candidate
            break
os.chdir(PROJECT_ROOT)

OUTPUT_DIR = Path("./explore/no0/form_metrics_260519")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CURVE_DIR = OUTPUT_DIR / "curves"
METRIC_DIR = OUTPUT_DIR / "metrics"
for folder in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, CURVE_DIR, METRIC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

n_bins = 10
min_count_per_bin = 20
frac = 0.18
poly_degree = 3
include_symbolic = True
n_grid = 300
random_state = 42
slope_tol = 0.03
EPS = 1e-12

METHOD_ORDER = [
    "Equal-width bins",
    "Equal-count bins",
    "LOWESS",
    "Polynomial degree 3",
    "GAM",
    "Piecewise linear",
    "Symbolic regression",
]

METHOD_COLORS = {
    "Equal-width bins": "#1f77b4",
    "Equal-count bins": "#ff7f0e",
    "LOWESS": "#2ca02c",
    "Polynomial degree 3": "#d62728",
    "GAM": "#9467bd",
    "Piecewise linear": "#8c564b",
    "Symbolic regression": "#e377c2",
}

METHOD_SHORT = {
    "Equal-width bins": "Equal-width",
    "Equal-count bins": "Equal-count",
    "LOWESS": "LOWESS",
    "Polynomial degree 3": "Poly-3",
    "GAM": "GAM",
    "Piecewise linear": "Piecewise",
    "Symbolic regression": "Symbolic",
}

zone_colors = {
    "HW": "#F27D7A",
    "HD": "#EFD447",
    "CW": "#95D98C",
    "CD": "#73AEEB",
}

zone_labels = {
    "HW": "WW - Warm-Wet",
    "HD": "WD - Warm-Dry",
    "CW": "CW - Cold-Wet",
    "CD": "CD - Cold-Dry",
}

full_label_map = {
    "P": "Precipitation (mm yr$^{-1}$)",
    "LAI": "Leaf Area Index",
    "T": "Transpiration (mm yr$^{-1}$)",
    "ET": "Evapotranspiration (mm yr$^{-1}$)",
    "E_veg": "Vegetation Evaporation (mm yr$^{-1}$)",
    "E_soil": "Soil Evaporation (mm yr$^{-1}$)",
}

short_label_map = {
    "P": "Precipitation",
    "LAI": "Leaf Area Index",
    "T": "Transpiration",
    "ET": "Evapotranspiration",
    "E_veg": "Vegetation Evaporation",
    "E_soil": "Soil Evaporation",
}

SHAPE_LABEL_THRESHOLDS = {
    "flat_abs_net_change": 0.05,
    "flat_fraction": 0.70,
    "segment_abs_slope_flat": 0.06,
    "linear_abs_corr": 0.95,
    "linear_corr_difference": 0.08,
    "linear_slope_iqr": 0.20,
    "monotonic_abs_spearman": 0.85,
    "monotonic_fraction": 0.80,
    "nonlinear_corr_gap": 0.10,
    "saturation_early_late_ratio": 2.0,
    "saturation_late_abs_slope": 0.08,
    "nonmonotonic_min_fraction": 0.20,
}

TARGET_ANALYSES = [
    {
        "analysis_group": "HW_T",
        "zones": ["HW"],
        "x_labels": ["P", "LAI"],
        "y_label": "T",
    },
    {
        "analysis_group": "all_zones_E_soil",
        "zones": ["HW", "HD", "CW", "CD"],
        "x_labels": ["P", "LAI"],
        "y_label": "E_soil",
    },
]

MODEL_SPECS = []

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


Project root: /Users/mimi/Documents/Code/Github/GSA-work
Output directory: explore/no0/form_metrics_260519


## 3. Data loading

Load the two model datasets using the same variable mapping and preferred data-reading logic as the current notebook, with fallback paths for running from the project root.


In [3]:
data_dir = Path("./preprocessed/preprocessed_with_zones")
if not data_dir.exists():
    candidate_data_dirs = [
        Path("./data/preprocessed/preprocessed_with_zones"),
        Path("./explore/no0/preprocessed/preprocessed_with_zones"),
        Path("./explore/data/preprocessed/preprocessed_with_zones"),
        Path("./explore/data/preprocessed/no0/preprocessed_with_zones"),
    ]
    for candidate in candidate_data_dirs:
        if candidate.exists():
            data_dir = candidate
            break

classic_df = pd.read_csv(data_dir / "classic_with_climate_zones_filtered_30yr_mean.csv")
lpj_df = pd.read_csv(data_dir / "lpj_guess_with_climate_zones_filtered_30yr_mean.csv")

var_map = {
    "P": "precipitation",
    "T": "tran",
    "LAI": "lai",
    "ET": "evapotrans",
    "E_veg": "evspsblveg",
    "E_soil": "evspsblsoi",
}

pairs = [
    ("P", "ET"),
    ("P", "T"),
    ("P", "E_veg"),
    ("P", "E_soil"),
    ("LAI", "ET"),
    ("LAI", "T"),
    ("LAI", "E_veg"),
    ("LAI", "E_soil"),
]

MODEL_SPECS = [
    ("CLASSIC", classic_df),
    ("LPJ-GUESS", lpj_df),
]

print(f"Data directory: {data_dir}")
print(f"CLASSIC rows: {len(classic_df):,}")
print(f"LPJ-GUESS rows: {len(lpj_df):,}")
print("CLASSIC zones:", classic_df["climate_zone"].value_counts(dropna=False).to_dict())
print("LPJ-GUESS zones:", lpj_df["climate_zone"].value_counts(dropna=False).to_dict())


Data directory: data/preprocessed/preprocessed_with_zones
CLASSIC rows: 58,293
LPJ-GUESS rows: 55,966
CLASSIC zones: {'CW': 21322, 'CD': 13445, 'HW': 12340, 'HD': 11147, 'Unknown': 39}
LPJ-GUESS zones: {'CW': 21472, 'HW': 12538, 'HD': 11341, 'CD': 10397, 'Unknown': 218}


## 4. Fitting method functions

Seven fitting methods are implemented with the same basic logic as the current notebook. Each method returns a curve or a logged failure reason.


In [4]:
def clean_xy(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]


def finalize_curve(x, y, min_points=2):
    x, y = clean_xy(x, y)
    if len(x) < min_points:
        return None, "fewer than two finite fitted points"

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    # Remove duplicate x coordinates by taking the median fitted y at each x.
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if df.empty:
        return None, "no finite fitted points after duplicate handling"
    df = df.groupby("x", as_index=False)["y"].median().sort_values("x")

    if len(df) < min_points:
        return None, "fewer than two unique fitted x values"
    return {"x": df["x"].to_numpy(), "y": df["y"].to_numpy()}, ""


def compute_equal_width_binned_medians(x, y, n_bins=10, min_count_per_bin=20):
    x, y = clean_xy(x, y)
    if len(x) < min_count_per_bin:
        return None, f"not enough finite observations: {len(x)}"

    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if not np.isfinite(x_min) or not np.isfinite(x_max) or abs(x_max - x_min) <= EPS:
        return None, "x has zero or non-finite range"

    edges = np.linspace(x_min, x_max, n_bins + 1)
    x_med = []
    y_med = []
    for i in range(n_bins):
        left = edges[i]
        right = edges[i + 1]
        if i == n_bins - 1:
            mask = (x >= left) & (x <= right)
        else:
            mask = (x >= left) & (x < right)
        if np.sum(mask) < min_count_per_bin:
            continue
        x_med.append(np.nanmedian(x[mask]))
        y_med.append(np.nanmedian(y[mask]))

    curve, reason = finalize_curve(x_med, y_med)
    if curve is None:
        return None, f"insufficient populated bins; {reason}"
    return curve, ""


def compute_equal_count_binned_medians(x, y, n_bins=10, min_count_per_bin=20):
    x, y = clean_xy(x, y)
    if len(x) < min_count_per_bin:
        return None, f"not enough finite observations: {len(x)}"

    order = np.argsort(x)
    x = x[order]
    y = y[order]
    bin_indices = np.array_split(np.arange(len(x)), n_bins)

    x_med = []
    y_med = []
    for idx in bin_indices:
        if len(idx) < min_count_per_bin:
            continue
        x_med.append(np.nanmedian(x[idx]))
        y_med.append(np.nanmedian(y[idx]))

    curve, reason = finalize_curve(x_med, y_med)
    if curve is None:
        return None, f"insufficient populated bins; {reason}"
    return curve, ""


def compute_lowess_curve(x, y, frac=0.18):
    if not LOWESS_AVAILABLE:
        return None, f"statsmodels LOWESS unavailable: {LOWESS_IMPORT_ERROR}"
    x, y = clean_xy(x, y)
    if len(x) < 5:
        return None, f"not enough finite observations for LOWESS: {len(x)}"
    try:
        smoothed = lowess(y, x, frac=frac, return_sorted=True)
        curve, reason = finalize_curve(smoothed[:, 0], smoothed[:, 1])
        if curve is None:
            return None, reason
        return curve, ""
    except Exception as exc:
        return None, repr(exc)


def compute_polynomial_curve(x, y, degree=3, n_grid=300):
    if not SKLEARN_AVAILABLE:
        return None, f"sklearn unavailable: {SKLEARN_IMPORT_ERROR}"
    x, y = clean_xy(x, y)
    if len(x) < degree + 2:
        return None, f"not enough finite observations for degree {degree}: {len(x)}"
    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if abs(x_max - x_min) <= EPS:
        return None, "x has zero range"
    try:
        x_grid = np.linspace(x_min, x_max, n_grid)
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        x_poly = poly.fit_transform(x.reshape(-1, 1))
        model = LinearRegression()
        model.fit(x_poly, y)
        y_grid = model.predict(poly.transform(x_grid.reshape(-1, 1)))
        curve, reason = finalize_curve(x_grid, y_grid)
        if curve is None:
            return None, reason
        return curve, ""
    except Exception as exc:
        return None, repr(exc)


def compute_gam_curve(x, y, n_grid=300, n_splines=12):
    if not PYGAM_AVAILABLE:
        return None, f"pygam unavailable: {PYGAM_IMPORT_ERROR}"
    x, y = clean_xy(x, y)
    if len(x) < 20:
        return None, f"not enough finite observations for GAM: {len(x)}"
    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if abs(x_max - x_min) <= EPS:
        return None, "x has zero range"
    try:
        x_grid = np.linspace(x_min, x_max, n_grid)
        gam = LinearGAM(s(0, n_splines=n_splines)).fit(x.reshape(-1, 1), y)
        y_grid = gam.predict(x_grid.reshape(-1, 1))
        curve, reason = finalize_curve(x_grid, y_grid)
        if curve is None:
            return None, reason
        return curve, ""
    except Exception as exc:
        return None, repr(exc)


def piecewise_linear_one_break(x, x0, k1, k2, b):
    return np.where(x < x0, k1 * x + b, k1 * x0 + b + k2 * (x - x0))


def compute_piecewise_linear_curve(x, y, n_grid=300):
    x, y = clean_xy(x, y)
    if len(x) < 20:
        return None, f"not enough finite observations for piecewise linear: {len(x)}"
    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if abs(x_max - x_min) <= EPS:
        return None, "x has zero range"
    try:
        x_grid = np.linspace(x_min, x_max, n_grid)
        p0 = [np.nanmedian(x), 0.0, 0.0, np.nanmedian(y)]
        bounds = ([x_min, -np.inf, -np.inf, -np.inf], [x_max, np.inf, np.inf, np.inf])
        params, _ = curve_fit(
            piecewise_linear_one_break,
            x,
            y,
            p0=p0,
            bounds=bounds,
            maxfev=20000,
        )
        y_grid = piecewise_linear_one_break(x_grid, *params)
        curve, reason = finalize_curve(x_grid, y_grid)
        if curve is None:
            return None, reason
        curve["breakpoint"] = float(params[0])
        return curve, ""
    except Exception as exc:
        return None, repr(exc)


def compute_symbolic_regression_curve(x, y, n_grid=300, max_train=8000, random_state=42):
    if not GPLEARN_AVAILABLE:
        return None, f"gplearn unavailable: {GPLEARN_IMPORT_ERROR}"
    x, y = clean_xy(x, y)
    if len(x) < 50:
        return None, f"not enough finite observations for symbolic regression: {len(x)}"
    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if abs(x_max - x_min) <= EPS:
        return None, "x has zero range"
    try:
        x_grid = np.linspace(x_min, x_max, n_grid)
        if len(x) > max_train:
            rng = np.random.default_rng(random_state)
            idx = rng.choice(len(x), size=max_train, replace=False)
            x_train = x[idx]
            y_train = y[idx]
        else:
            x_train = x
            y_train = y

        est = SymbolicRegressor(
            population_size=1000,
            generations=15,
            stopping_criteria=0.01,
            p_crossover=0.7,
            p_subtree_mutation=0.1,
            p_hoist_mutation=0.05,
            p_point_mutation=0.1,
            max_samples=0.8,
            verbose=0,
            parsimony_coefficient=0.01,
            random_state=random_state,
            function_set=("add", "sub", "mul", "div", "sqrt", "log"),
        )
        est.fit(x_train.reshape(-1, 1), y_train)
        y_grid = est.predict(x_grid.reshape(-1, 1))
        curve, reason = finalize_curve(x_grid, y_grid)
        if curve is None:
            return None, reason
        curve["expression"] = str(est._program)
        return curve, ""
    except Exception as exc:
        return None, repr(exc)


def compute_all_curves_for_relation(
    x,
    y,
    n_bins=10,
    min_count_per_bin=20,
    frac=0.18,
    poly_degree=3,
    include_symbolic=True,
    n_grid=300,
    random_state=42,
):
    method_calls = {
        "Equal-width bins": lambda: compute_equal_width_binned_medians(x, y, n_bins, min_count_per_bin),
        "Equal-count bins": lambda: compute_equal_count_binned_medians(x, y, n_bins, min_count_per_bin),
        "LOWESS": lambda: compute_lowess_curve(x, y, frac),
        f"Polynomial degree {poly_degree}": lambda: compute_polynomial_curve(x, y, poly_degree, n_grid),
        "GAM": lambda: compute_gam_curve(x, y, n_grid),
        "Piecewise linear": lambda: compute_piecewise_linear_curve(x, y, n_grid),
        "Symbolic regression": lambda: compute_symbolic_regression_curve(x, y, n_grid, 8000, random_state),
    }

    curves = {}
    log_rows = []
    for method in METHOD_ORDER:
        if method == "Symbolic regression" and not include_symbolic:
            log_rows.append({"method": method, "status": "skipped", "reason": "include_symbolic is False"})
            continue
        try:
            curve, reason = method_calls[method]()
        except Exception as exc:
            curve, reason = None, repr(exc)
        if curve is None:
            status = "unavailable" if "unavailable" in str(reason).lower() else "failed"
            log_rows.append({"method": method, "status": status, "reason": reason})
        else:
            curves[method] = curve
            log_rows.append({"method": method, "status": "success", "reason": ""})
    return curves, log_rows


## 5. Metric utility functions

Shared helpers clean fitted curves, normalize values, summarize slopes, interpolate predictions, and save figures consistently.


In [5]:
def finite_or_nan(value):
    try:
        value = float(value)
    except Exception:
        return np.nan
    return value if np.isfinite(value) else np.nan


def safe_corr(func, x, y):
    x, y = clean_xy(x, y)
    if len(x) < 3:
        return np.nan, np.nan
    if np.nanstd(x) <= EPS or np.nanstd(y) <= EPS:
        return np.nan, np.nan
    try:
        result = func(x, y)
        if hasattr(result, "statistic"):
            return finite_or_nan(result.statistic), finite_or_nan(result.pvalue)
        return finite_or_nan(result[0]), finite_or_nan(result[1])
    except Exception:
        return np.nan, np.nan


def minmax_normalize(values, eps=EPS):
    arr = np.asarray(values, dtype=float)
    valid = np.isfinite(arr)
    out = np.full(arr.shape, np.nan, dtype=float)
    if not np.any(valid):
        return out, np.nan, np.nan, np.nan
    vmin = np.nanmin(arr[valid])
    vmax = np.nanmax(arr[valid])
    vrange = vmax - vmin
    if not np.isfinite(vrange) or vrange <= eps:
        return out, vmin, vmax, vrange
    out[valid] = (arr[valid] - vmin) / vrange
    return out, vmin, vmax, vrange


def prepare_curve_arrays(x_fit, y_fit):
    curve, reason = finalize_curve(x_fit, y_fit)
    if curve is None:
        return np.array([]), np.array([])
    return curve["x"], curve["y"]


def compute_local_slopes(x_norm, y_norm, eps=EPS):
    x_norm = np.asarray(x_norm, dtype=float)
    y_norm = np.asarray(y_norm, dtype=float)
    valid = np.isfinite(x_norm) & np.isfinite(y_norm)
    x_norm = x_norm[valid]
    y_norm = y_norm[valid]
    if len(x_norm) < 2:
        return np.array([]), np.array([]), np.array([])
    dx = np.diff(x_norm)
    dy = np.diff(y_norm)
    valid_seg = np.isfinite(dx) & np.isfinite(dy) & (np.abs(dx) > eps)
    slopes = dy[valid_seg] / dx[valid_seg]
    x_mid = 0.5 * (x_norm[:-1][valid_seg] + x_norm[1:][valid_seg])
    dy_valid = dy[valid_seg]
    finite = np.isfinite(slopes) & np.isfinite(x_mid)
    return slopes[finite], x_mid[finite], dy_valid[finite]


def slope_sign_value(value, tol=slope_tol):
    if not np.isfinite(value):
        return np.nan
    if value > tol:
        return 1
    if value < -tol:
        return -1
    return 0


def segment_mean(x_pos, values, lo, hi, include_left=True):
    x_pos = np.asarray(x_pos, dtype=float)
    values = np.asarray(values, dtype=float)
    if include_left:
        mask = (x_pos >= lo) & (x_pos <= hi)
    else:
        mask = (x_pos > lo) & (x_pos <= hi)
    mask &= np.isfinite(values) & np.isfinite(x_pos)
    if not np.any(mask):
        return np.nan
    return finite_or_nan(np.nanmean(values[mask]))


def segment_quantile_width(values, low, high):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    return finite_or_nan(np.nanpercentile(values, high) - np.nanpercentile(values, low))


def safe_iqr(values):
    return segment_quantile_width(values, 25, 75)


def save_figure(fig, stem):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    try:
        fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    except Exception as exc:
        print(f"PDF save failed for {pdf_path}: {exc}")
    plt.close(fig)


def style_axis(ax):
    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(True, which="major", color="0.82", linestyle="-", linewidth=0.8)
    ax.minorticks_off()
    for side in ["top", "right", "bottom", "left"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.9)
        ax.spines[side].set_color("0.35")
    ax.tick_params(axis="both", which="major", labelsize=10, length=4, width=0.8, color="0.35")


def get_relation_data(df, zone, x_label, y_label):
    x_col = var_map[x_label]
    y_col = var_map[y_label]
    sub = (
        df.loc[df["climate_zone"] == zone, [x_col, y_col]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    return sub[x_col].to_numpy(), sub[y_col].to_numpy(), sub


## 6. Shape metrics

Shape metrics describe the structural pattern of each fitted response curve, including monotonicity, curvature proxies, slope sign fractions, and early/middle/late behavior.


In [6]:
def detect_turning_points(x_norm, y_norm, slopes, x_mid, tol=slope_tol, edge_exclusion=0.02):
    result = {
        "has_internal_valley": False,
        "has_internal_peak": False,
        "number_of_internal_extrema": 0,
        "turning_point_x_norm": np.nan,
        "turning_point_y_norm": np.nan,
        "turning_point_type": np.nan,
    }
    if len(slopes) < 2:
        return result

    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nonflat_idx = np.where(signs != 0)[0]
    if len(nonflat_idx) < 2:
        return result

    candidates = []
    for i in range(len(nonflat_idx) - 1):
        left_idx = nonflat_idx[i]
        right_idx = nonflat_idx[i + 1]
        s1 = signs[left_idx]
        s2 = signs[right_idx]
        if s1 == s2:
            continue
        turn_x = 0.5 * (x_mid[left_idx] + x_mid[right_idx])
        if not np.isfinite(turn_x) or turn_x <= edge_exclusion or turn_x >= 1 - edge_exclusion:
            continue
        turn_y = np.interp(turn_x, x_norm, y_norm)
        if s1 < 0 and s2 > 0:
            turn_type = "valley"
        elif s1 > 0 and s2 < 0:
            turn_type = "peak"
        else:
            turn_type = "unknown"
        candidates.append({"x": turn_x, "y": turn_y, "type": turn_type})

    if not candidates:
        return result

    result["has_internal_valley"] = any(c["type"] == "valley" for c in candidates)
    result["has_internal_peak"] = any(c["type"] == "peak" for c in candidates)
    result["number_of_internal_extrema"] = len(candidates)

    scores = []
    for cand in candidates:
        endpoint_contrast = min(abs(cand["y"] - y_norm[0]), abs(cand["y"] - y_norm[-1]))
        scores.append(endpoint_contrast)
    best = candidates[int(np.nanargmax(scores))]
    result["turning_point_x_norm"] = finite_or_nan(best["x"])
    result["turning_point_y_norm"] = finite_or_nan(best["y"])
    result["turning_point_type"] = best["type"]
    return result


def compute_two_line_proxy(x_norm, y_norm, turning_point_x_norm):
    out = {
        "two_line_left_slope": np.nan,
        "two_line_right_slope": np.nan,
        "two_line_left_p": np.nan,
        "two_line_right_p": np.nan,
        "two_line_test_evidence": np.nan,
        "u_test_stat": np.nan,
        "u_test_p": np.nan,
        "lind_mehlum_stat": np.nan,
        "lind_mehlum_p": np.nan,
    }
    if not np.isfinite(turning_point_x_norm):
        return out

    left = np.isfinite(x_norm) & np.isfinite(y_norm) & (x_norm <= turning_point_x_norm)
    right = np.isfinite(x_norm) & np.isfinite(y_norm) & (x_norm >= turning_point_x_norm)
    if left.sum() >= 3:
        lr = stats.linregress(x_norm[left], y_norm[left])
        out["two_line_left_slope"] = finite_or_nan(lr.slope)
        out["two_line_left_p"] = finite_or_nan(lr.pvalue)
    if right.sum() >= 3:
        rr = stats.linregress(x_norm[right], y_norm[right])
        out["two_line_right_slope"] = finite_or_nan(rr.slope)
        out["two_line_right_p"] = finite_or_nan(rr.pvalue)

    left_slope = out["two_line_left_slope"]
    right_slope = out["two_line_right_slope"]
    left_p = out["two_line_left_p"]
    right_p = out["two_line_right_p"]
    if np.isfinite(left_slope) and np.isfinite(right_slope):
        opposite = (left_slope < -slope_tol and right_slope > slope_tol) or (left_slope > slope_tol and right_slope < -slope_tol)
        p_ok = (not np.isfinite(left_p) or left_p <= 0.05) and (not np.isfinite(right_p) or right_p <= 0.05)
        out["two_line_test_evidence"] = bool(opposite and p_ok)
    return out


def compute_shape_metrics(x_fit, y_fit):
    x, y = prepare_curve_arrays(x_fit, y_fit)
    metrics = {
        "n_curve_points": len(x),
    }
    if len(x) < 2:
        return metrics

    x_norm, x_min, x_max, x_range = minmax_normalize(x)
    y_norm, y_min, y_max, y_range = minmax_normalize(y)

    pearson_r, pearson_p = safe_corr(pearsonr, x, y)
    spearman_rho, spearman_p = safe_corr(spearmanr, x, y)
    kendall_tau, kendall_p = safe_corr(kendalltau, x, y)

    metrics.update({
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "kendall_tau": kendall_tau,
        "kendall_p": kendall_p,
        "abs_pearson_r": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
        "abs_spearman_r": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
        "abs_spearman_minus_abs_pearson": (abs(spearman_rho) - abs(pearson_r)) if np.isfinite(spearman_rho) and np.isfinite(pearson_r) else np.nan,
        "raw_net_change": finite_or_nan(y[-1] - y[0]),
        "absolute_net_change": finite_or_nan(abs(y[-1] - y[0])),
        "normalized_net_change": finite_or_nan(y_norm[-1] - y_norm[0]) if np.all(np.isfinite([y_norm[0], y_norm[-1]])) else np.nan,
        "normalized_absolute_net_change": finite_or_nan(abs(y_norm[-1] - y_norm[0])) if np.all(np.isfinite([y_norm[0], y_norm[-1]])) else np.nan,
    })

    slopes, x_mid, _ = compute_local_slopes(x_norm, y_norm)
    if len(slopes) == 0:
        metrics.update({
            "positive_slope_fraction": np.nan,
            "negative_slope_fraction": np.nan,
            "flat_slope_fraction": np.nan,
            "dominant_slope_sign": np.nan,
            "monotonic_fraction": np.nan,
            "number_of_slope_sign_changes": np.nan,
        })
        return metrics

    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > slope_tol] = 1
    signs[slopes < -slope_tol] = -1
    positive_fraction = np.mean(signs == 1)
    negative_fraction = np.mean(signs == -1)
    flat_fraction = np.mean(signs == 0)
    nonzero_signs = signs[signs != 0]
    sign_changes = int(np.sum(nonzero_signs[1:] != nonzero_signs[:-1])) if len(nonzero_signs) >= 2 else 0
    if flat_fraction >= max(positive_fraction, negative_fraction):
        dominant = "flat"
    elif positive_fraction >= negative_fraction:
        dominant = "positive"
    else:
        dominant = "negative"

    early_slope = segment_mean(x_mid, slopes, 0.0, 1 / 3)
    middle_slope = segment_mean(x_mid, slopes, 1 / 3, 2 / 3, include_left=False)
    late_slope = segment_mean(x_mid, slopes, 2 / 3, 1.0, include_left=False)
    early_sign = slope_sign_value(early_slope)
    late_sign = slope_sign_value(late_slope)

    metrics.update({
        "positive_slope_fraction": finite_or_nan(positive_fraction),
        "negative_slope_fraction": finite_or_nan(negative_fraction),
        "flat_slope_fraction": finite_or_nan(flat_fraction),
        "dominant_slope_sign": dominant,
        "monotonic_fraction": finite_or_nan(max(positive_fraction, negative_fraction)),
        "number_of_slope_sign_changes": sign_changes,
        "early_slope": early_slope,
        "middle_slope": middle_slope,
        "late_slope": late_slope,
        "early_absolute_slope": finite_or_nan(abs(early_slope)) if np.isfinite(early_slope) else np.nan,
        "middle_absolute_slope": finite_or_nan(abs(middle_slope)) if np.isfinite(middle_slope) else np.nan,
        "late_absolute_slope": finite_or_nan(abs(late_slope)) if np.isfinite(late_slope) else np.nan,
        "early_late_slope_sign_consistency": bool(early_sign == late_sign and early_sign != 0) if np.isfinite(early_sign) and np.isfinite(late_sign) else np.nan,
        "early_late_slope_sign_opposition": bool(early_sign * late_sign < 0) if np.isfinite(early_sign) and np.isfinite(late_sign) else np.nan,
    })

    turning = detect_turning_points(x_norm, y_norm, slopes, x_mid)
    metrics.update(turning)
    metrics["turning_point_x"] = np.nan
    if np.isfinite(metrics.get("turning_point_x_norm", np.nan)) and np.isfinite(x_min) and np.isfinite(x_range):
        metrics["turning_point_x"] = finite_or_nan(x_min + metrics["turning_point_x_norm"] * x_range)
    metrics.update(compute_two_line_proxy(x_norm, y_norm, metrics.get("turning_point_x_norm", np.nan)))
    return metrics


## 7. Strength metrics

Strength metrics quantify the response magnitude with amplitude, net change, variation, elasticity, normalized slopes, and accumulated slope summaries.


In [7]:
def safe_ratio(a, b):
    if np.isfinite(a) and np.isfinite(b) and abs(b) > EPS:
        return finite_or_nan(a / b)
    return np.nan


def accumulated_absolute_variation_values(x_norm, y_norm):
    """Cumulative absolute response variation on normalized fitted y.

    This is equivalent to cumsum(abs(local_slope) * dx_norm), but avoids
    point-count sensitivity from summing unweighted local slopes.
    """
    x_norm = np.asarray(x_norm, dtype=float)
    y_norm = np.asarray(y_norm, dtype=float)
    valid = np.isfinite(x_norm) & np.isfinite(y_norm)
    x_norm = x_norm[valid]
    y_norm = y_norm[valid]
    if len(x_norm) < 2:
        return np.array([]), np.array([])

    dx = np.diff(x_norm)
    dy = np.diff(y_norm)
    valid_seg = np.isfinite(dx) & np.isfinite(dy) & (np.abs(dx) > EPS)
    if not np.any(valid_seg):
        return np.array([]), np.array([])

    x_mid = 0.5 * (x_norm[:-1][valid_seg] + x_norm[1:][valid_seg])
    accumulated = np.cumsum(np.abs(dy[valid_seg]))
    return x_mid, accumulated


def value_at_or_before(x_pos, values, threshold):
    x_pos = np.asarray(x_pos, dtype=float)
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(x_pos) & np.isfinite(values) & (x_pos <= threshold)
    if not np.any(mask):
        return np.nan
    return finite_or_nan(values[mask][-1])


def compute_strength_metrics(x_fit, y_fit, y_raw):
    x, y = prepare_curve_arrays(x_fit, y_fit)
    metrics = {}
    if len(x) < 2:
        return metrics

    x_norm, x_min, x_max, x_range = minmax_normalize(x)
    y_norm, y_min, y_max, y_range = minmax_normalize(y)
    _, raw_y_min, raw_y_max, raw_y_range = minmax_normalize(y_raw)
    y_raw_clean = np.asarray(y_raw, dtype=float)
    y_raw_clean = y_raw_clean[np.isfinite(y_raw_clean)]
    raw_y_sd = finite_or_nan(np.nanstd(y_raw_clean, ddof=1)) if len(y_raw_clean) > 1 else np.nan

    raw_amplitude = finite_or_nan(np.nanmax(y) - np.nanmin(y))
    amplitude_norm_by_y_sd = safe_ratio(raw_amplitude, raw_y_sd)
    amplitude_norm_by_y_range = safe_ratio(raw_amplitude, raw_y_range)
    # Keep this column name for continuity, but define it as the SD-normalized amplitude.
    normalized_amplitude = amplitude_norm_by_y_sd

    raw_total_variation = finite_or_nan(np.sum(np.abs(np.diff(y))))
    normalized_total_variation = finite_or_nan(np.sum(np.abs(np.diff(y_norm)))) if np.all(np.isfinite(y_norm)) else np.nan
    raw_net_change = finite_or_nan(y[-1] - y[0])
    normalized_net_change = finite_or_nan(y_norm[-1] - y_norm[0]) if np.all(np.isfinite([y_norm[0], y_norm[-1]])) else np.nan

    slopes_norm, x_mid_norm, _ = compute_local_slopes(x_norm, y_norm)
    slopes_raw, x_mid_raw, _ = compute_local_slopes(x, y)

    elasticity_values = []
    if len(slopes_raw) > 0:
        x_mid_raw_el = x_mid_raw
        y_mid_raw = np.interp(x_mid_raw_el, x, y)
        with np.errstate(divide="ignore", invalid="ignore"):
            elasticity = slopes_raw * (x_mid_raw_el / y_mid_raw)
        valid_elasticity = np.isfinite(elasticity) & np.isfinite(x_mid_raw_el) & np.isfinite(y_mid_raw) & (np.abs(y_mid_raw) > EPS) & (np.abs(x_mid_raw_el) > EPS)
        elasticity_values = np.abs(elasticity[valid_elasticity])

    early_slope = segment_mean(x_mid_norm, slopes_norm, 0.0, 1 / 3)
    middle_slope = segment_mean(x_mid_norm, slopes_norm, 1 / 3, 2 / 3, include_left=False)
    late_slope = segment_mean(x_mid_norm, slopes_norm, 2 / 3, 1.0, include_left=False)

    early_abs_slope = finite_or_nan(abs(early_slope)) if np.isfinite(early_slope) else np.nan
    middle_abs_slope = finite_or_nan(abs(middle_slope)) if np.isfinite(middle_slope) else np.nan
    late_abs_slope = finite_or_nan(abs(late_slope)) if np.isfinite(late_slope) else np.nan

    acc_x, acc_values = accumulated_absolute_variation_values(x_norm, y_norm)

    metrics.update({
        "raw_amplitude": raw_amplitude,
        "normalized_amplitude": normalized_amplitude,
        "amplitude_norm_by_y_sd": amplitude_norm_by_y_sd,
        "amplitude_norm_by_y_range": amplitude_norm_by_y_range,
        "amplitude_divided_by_raw_y_range": amplitude_norm_by_y_range,
        "raw_y_sd_for_strength": raw_y_sd,
        "strength_raw_net_change": raw_net_change,
        "strength_absolute_net_change": finite_or_nan(abs(raw_net_change)) if np.isfinite(raw_net_change) else np.nan,
        "strength_normalized_net_change": normalized_net_change,
        "strength_normalized_absolute_net_change": finite_or_nan(abs(normalized_net_change)) if np.isfinite(normalized_net_change) else np.nan,
        "net_change_by_sd": safe_ratio(raw_net_change, raw_y_sd),
        "absolute_net_change_by_sd": safe_ratio(abs(raw_net_change), raw_y_sd) if np.isfinite(raw_net_change) else np.nan,
        "abs_net_change_by_sd": safe_ratio(abs(raw_net_change), raw_y_sd) if np.isfinite(raw_net_change) else np.nan,
        "raw_total_variation": raw_total_variation,
        "total_variation_by_sd": safe_ratio(raw_total_variation, raw_y_sd),
        "normalized_total_variation": normalized_total_variation,
        "mean_absolute_elasticity": finite_or_nan(np.nanmean(elasticity_values)) if len(elasticity_values) else np.nan,
        "median_absolute_elasticity": finite_or_nan(np.nanmedian(elasticity_values)) if len(elasticity_values) else np.nan,
        "max_absolute_elasticity": finite_or_nan(np.nanmax(elasticity_values)) if len(elasticity_values) else np.nan,
        "mean_absolute_slope": finite_or_nan(np.nanmean(np.abs(slopes_norm))) if len(slopes_norm) else np.nan,
        "max_absolute_slope": finite_or_nan(np.nanmax(np.abs(slopes_norm))) if len(slopes_norm) else np.nan,
        "median_absolute_slope": finite_or_nan(np.nanmedian(np.abs(slopes_norm))) if len(slopes_norm) else np.nan,
        "slope_standard_deviation": finite_or_nan(np.nanstd(slopes_norm)) if len(slopes_norm) else np.nan,
        "slope_interquartile_range": safe_iqr(slopes_norm) if len(slopes_norm) else np.nan,
        "overall_standardized_slope": finite_or_nan(np.nanmean(slopes_norm)) if len(slopes_norm) else np.nan,
        "mean_standardized_absolute_slope": finite_or_nan(np.nanmean(np.abs(slopes_norm))) if len(slopes_norm) else np.nan,
        "max_standardized_absolute_slope": finite_or_nan(np.nanmax(np.abs(slopes_norm))) if len(slopes_norm) else np.nan,
        "early_standardized_slope": early_slope,
        "middle_standardized_slope": middle_slope,
        "late_standardized_slope": late_slope,
        "early_absolute_standardized_slope": early_abs_slope,
        "middle_absolute_standardized_slope": middle_abs_slope,
        "late_absolute_standardized_slope": late_abs_slope,
        "middle_to_early_absolute_slope_ratio": safe_ratio(middle_abs_slope, early_abs_slope),
        "late_to_early_absolute_slope_ratio": safe_ratio(late_abs_slope, early_abs_slope),
        "accumulated_absolute_variation_final_value": finite_or_nan(acc_values[-1]) if len(acc_values) else np.nan,
        "accumulated_absolute_variation_early_value": value_at_or_before(acc_x, acc_values, 1 / 3) if len(acc_values) else np.nan,
        "accumulated_absolute_variation_middle_value": value_at_or_before(acc_x, acc_values, 2 / 3) if len(acc_values) else np.nan,
        "accumulated_absolute_variation_late_value": finite_or_nan(acc_values[-1]) if len(acc_values) else np.nan,
        "accumulated_slope_final_value": finite_or_nan(acc_values[-1]) if len(acc_values) else np.nan,
        "accumulated_slope_early_value": value_at_or_before(acc_x, acc_values, 1 / 3) if len(acc_values) else np.nan,
        "accumulated_slope_middle_value": value_at_or_before(acc_x, acc_values, 2 / 3) if len(acc_values) else np.nan,
        "accumulated_slope_late_value": finite_or_nan(acc_values[-1]) if len(acc_values) else np.nan,
    })
    return metrics

## 8. Tightness metrics

Tightness metrics use one residual definition for every method: raw `y` minus the fitted curve interpolated back to the raw `x` positions.


In [8]:
def interpolate_predictions(x_raw, y_raw, x_fit, y_fit):
    x_fit, y_fit = prepare_curve_arrays(x_fit, y_fit)
    x_raw, y_raw = clean_xy(x_raw, y_raw)
    if len(x_fit) < 2 or len(x_raw) == 0:
        return np.array([]), np.array([]), np.array([])
    in_range = (x_raw >= np.nanmin(x_fit)) & (x_raw <= np.nanmax(x_fit))
    in_range &= np.isfinite(x_raw) & np.isfinite(y_raw)
    x_eval = x_raw[in_range]
    y_eval = y_raw[in_range]
    if len(x_eval) == 0:
        return np.array([]), np.array([]), np.array([])
    y_pred = np.interp(x_eval, x_fit, y_fit)
    valid = np.isfinite(y_pred) & np.isfinite(y_eval)
    return x_eval[valid], y_eval[valid], y_pred[valid]


def summarize_residual_segment(resid):
    resid = np.asarray(resid, dtype=float)
    resid = resid[np.isfinite(resid)]
    if len(resid) == 0:
        return {
            "mean_absolute_residual": np.nan,
            "median_absolute_residual": np.nan,
            "rmse": np.nan,
            "residual_variance": np.nan,
            "buffer_width_90": np.nan,
            "buffer_width_95": np.nan,
        }
    return {
        "mean_absolute_residual": finite_or_nan(np.nanmean(np.abs(resid))),
        "median_absolute_residual": finite_or_nan(np.nanmedian(np.abs(resid))),
        "rmse": finite_or_nan(np.sqrt(np.nanmean(resid ** 2))),
        "residual_variance": finite_or_nan(np.nanvar(resid, ddof=1)) if len(resid) > 1 else 0.0,
        "buffer_width_90": segment_quantile_width(resid, 5, 95),
        "buffer_width_95": segment_quantile_width(resid, 2.5, 97.5),
    }


def compute_tightness_metrics(x_raw, y_raw, x_fit, y_fit):
    x_raw_clean, y_raw_clean = clean_xy(x_raw, y_raw)
    raw_y_sd = finite_or_nan(np.nanstd(y_raw_clean, ddof=1)) if len(y_raw_clean) > 1 else np.nan
    raw_y_range = finite_or_nan(np.nanmax(y_raw_clean) - np.nanmin(y_raw_clean)) if len(y_raw_clean) else np.nan

    x_eval, y_eval, y_pred = interpolate_predictions(x_raw_clean, y_raw_clean, x_fit, y_fit)
    metrics = {
        "tightness_n_raw": len(y_raw_clean),
        "tightness_n_predicted": len(y_pred),
        "tightness_prediction_fraction": safe_ratio(len(y_pred), len(y_raw_clean)),
        "raw_y_sd_for_tightness": raw_y_sd,
        "raw_y_range_for_tightness": raw_y_range,
    }
    if len(y_pred) == 0:
        return metrics

    residual = y_eval - y_pred
    sse = np.nansum(residual ** 2)
    sst = np.nansum((y_eval - np.nanmean(y_eval)) ** 2)
    rmse = finite_or_nan(np.sqrt(np.nanmean(residual ** 2)))
    residual_variance = finite_or_nan(np.nanvar(residual, ddof=1)) if len(residual) > 1 else 0.0
    residual_sd = finite_or_nan(np.nanstd(residual, ddof=1)) if len(residual) > 1 else 0.0
    residual_iqr = safe_iqr(residual)
    residual_90 = segment_quantile_width(residual, 5, 95)
    residual_95 = segment_quantile_width(residual, 2.5, 97.5)
    mean_abs_residual = finite_or_nan(np.nanmean(np.abs(residual)))
    median_abs_residual = finite_or_nan(np.nanmedian(np.abs(residual)))

    metrics.update({
        "r_squared": finite_or_nan(1 - sse / sst) if np.isfinite(sst) and sst > EPS else np.nan,
        "rmse": rmse,
        "normalized_rmse_raw_y_range": safe_ratio(rmse, raw_y_range),
        "normalized_rmse_raw_y_std": safe_ratio(rmse, raw_y_sd),
        "nrmse_by_raw_y_range": safe_ratio(rmse, raw_y_range),
        "nrmse_by_sd": safe_ratio(rmse, raw_y_sd),
        "residual_variance": residual_variance,
        "residual_standard_deviation": residual_sd,
        "mean_absolute_residual": mean_abs_residual,
        "median_absolute_residual": median_abs_residual,
        "residual_iqr": residual_iqr,
        "residual_90_interval_width": residual_90,
        "residual_95_interval_width": residual_95,
        "residual_90_interval_width_by_sd": safe_ratio(residual_90, raw_y_sd),
        "residual_95_interval_width_by_sd": safe_ratio(residual_95, raw_y_sd),
        "mean_buffer_width": mean_abs_residual,
        "median_buffer_width": median_abs_residual,
        "buffer_width_90": residual_90,
        "buffer_width_95": residual_95,
        "mean_buffer_width_by_sd": safe_ratio(mean_abs_residual, raw_y_sd),
        "median_buffer_width_by_sd": safe_ratio(median_abs_residual, raw_y_sd),
        "buffer_width90_by_sd": safe_ratio(residual_90, raw_y_sd),
        "buffer_width95_by_sd": safe_ratio(residual_95, raw_y_sd),
    })

    x_norm_eval, _, _, x_eval_range = minmax_normalize(x_eval)
    segments = {
        "early": (0.0, 1 / 3, True),
        "middle": (1 / 3, 2 / 3, False),
        "late": (2 / 3, 1.0, False),
    }
    for segment, (lo, hi, include_left) in segments.items():
        if not np.isfinite(x_eval_range) or x_eval_range <= EPS:
            seg_resid = np.array([])
        else:
            if include_left:
                mask = (x_norm_eval >= lo) & (x_norm_eval <= hi)
            else:
                mask = (x_norm_eval > lo) & (x_norm_eval <= hi)
            seg_resid = residual[mask]
        seg = summarize_residual_segment(seg_resid)
        seg_rmse_by_sd = safe_ratio(seg["rmse"], raw_y_sd)
        seg_buffer90_by_sd = safe_ratio(seg["buffer_width_90"], raw_y_sd)
        seg_buffer95_by_sd = safe_ratio(seg["buffer_width_95"], raw_y_sd)
        metrics.update({
            f"{segment}_mean_absolute_residual": seg["mean_absolute_residual"],
            f"{segment}_median_absolute_residual": seg["median_absolute_residual"],
            f"{segment}_rmse": seg["rmse"],
            f"{segment}_rmse_by_sd": seg_rmse_by_sd,
            f"{segment}_nrmse_by_sd": seg_rmse_by_sd,
            f"{segment}_residual_variance": seg["residual_variance"],
            f"{segment}_buffer_width_90": seg["buffer_width_90"],
            f"{segment}_buffer_width_95": seg["buffer_width_95"],
            f"{segment}_buffer_width90_by_sd": seg_buffer90_by_sd,
            f"{segment}_buffer_width95_by_sd": seg_buffer95_by_sd,
        })
    return metrics

## 9. Optional shape classification

The derived shape label is rule-based and uses thresholds defined in the configuration section. The candidate metrics remain the main output.


In [9]:
def classify_shape_label(row, thresholds=SHAPE_LABEL_THRESHOLDS):
    abs_net = row.get("normalized_absolute_net_change", np.nan)
    flat_fraction = row.get("flat_slope_fraction", np.nan)
    early_abs = row.get("early_absolute_slope", np.nan)
    middle_abs = row.get("middle_absolute_slope", np.nan)
    late_abs = row.get("late_absolute_slope", np.nan)
    abs_p = row.get("abs_pearson_r", np.nan)
    abs_s = row.get("abs_spearman_r", np.nan)
    corr_gap = abs(abs_s - abs_p) if np.isfinite(abs_s) and np.isfinite(abs_p) else np.nan
    monotonic_fraction = row.get("monotonic_fraction", np.nan)
    pos_fraction = row.get("positive_slope_fraction", np.nan)
    neg_fraction = row.get("negative_slope_fraction", np.nan)
    slope_iqr = row.get("slope_interquartile_range", np.nan)
    has_turn = bool(row.get("has_internal_valley", False)) or bool(row.get("has_internal_peak", False))
    opposition = bool(row.get("early_late_slope_sign_opposition", False))

    segment_abs = [early_abs, middle_abs, late_abs]
    all_segments_flat = all(np.isfinite(v) and v <= thresholds["segment_abs_slope_flat"] for v in segment_abs)

    if (
        np.isfinite(abs_net)
        and abs_net <= thresholds["flat_abs_net_change"]
        and np.isfinite(flat_fraction)
        and flat_fraction >= thresholds["flat_fraction"]
        and all_segments_flat
    ):
        return "Flat / no response"

    if (
        np.isfinite(abs_p)
        and np.isfinite(abs_s)
        and abs_p >= thresholds["linear_abs_corr"]
        and abs_s >= thresholds["linear_abs_corr"]
        and np.isfinite(corr_gap)
        and corr_gap <= thresholds["linear_corr_difference"]
        and np.isfinite(slope_iqr)
        and slope_iqr <= thresholds["linear_slope_iqr"]
    ):
        return "Linear"

    if (
        has_turn
        or opposition
        or (
            np.isfinite(pos_fraction)
            and np.isfinite(neg_fraction)
            and pos_fraction >= thresholds["nonmonotonic_min_fraction"]
            and neg_fraction >= thresholds["nonmonotonic_min_fraction"]
        )
    ):
        return "Non-monotonic: U-shaped / inverted-U"

    if (
        np.isfinite(abs_s)
        and abs_s >= thresholds["monotonic_abs_spearman"]
        and np.isfinite(monotonic_fraction)
        and monotonic_fraction >= thresholds["monotonic_fraction"]
    ):
        if (
            np.isfinite(early_abs)
            and np.isfinite(late_abs)
            and early_abs >= thresholds["saturation_early_late_ratio"] * max(late_abs, EPS)
            and late_abs <= thresholds["saturation_late_abs_slope"]
        ):
            return "Saturating monotonic"
        if (np.isfinite(corr_gap) and corr_gap >= thresholds["nonlinear_corr_gap"]) or not np.isfinite(slope_iqr) or slope_iqr > thresholds["linear_slope_iqr"]:
            return "Monotonic nonlinear"
        return "Linear"

    return "Complex nonlinear"


## 10. Batch analysis runner

Run every requested zone, model, input, output, and method combination. A failure in one fit is recorded and does not interrupt the remaining run.


In [10]:
def make_curve_records(zone, model, x_label, y_label, method, curve):
    x, y = prepare_curve_arrays(curve["x"], curve["y"])
    x_norm, _, _, _ = minmax_normalize(x)
    y_norm, _, _, _ = minmax_normalize(y)

    accumulated = np.full(len(x), np.nan)
    if len(x) >= 2 and np.all(np.isfinite(x_norm)) and np.all(np.isfinite(y_norm)):
        dx = np.diff(x_norm)
        dy = np.diff(y_norm)
        valid_seg = np.isfinite(dx) & np.isfinite(dy) & (np.abs(dx) > EPS)
        if len(valid_seg) == len(x) - 1 and np.all(valid_seg):
            accumulated = np.concatenate([[0.0], np.cumsum(np.abs(dy))])

    records = []
    for xi, yi, xni, yni, acc in zip(x, y, x_norm, y_norm, accumulated):
        records.append({
            "zone": zone,
            "model": model,
            "x_label": x_label,
            "y_label": y_label,
            "method": method,
            "x_fit": finite_or_nan(xi),
            "y_fit": finite_or_nan(yi),
            "x_fit_norm": finite_or_nan(xni),
            "y_fit_norm": finite_or_nan(yni),
            "accumulated_slope": finite_or_nan(acc),
            "accumulated_absolute_variation": finite_or_nan(acc),
        })
    return records


def run_batch_analysis():
    relation_results = {}
    metric_records = []
    curve_records = []
    log_records = []

    for target in TARGET_ANALYSES:
        analysis_group = target["analysis_group"]
        y_label = target["y_label"]
        for zone in target["zones"]:
            for x_label in target["x_labels"]:
                for model_name, df in MODEL_SPECS:
                    x_raw, y_raw, sub = get_relation_data(df, zone, x_label, y_label)
                    print(f"Fitting {analysis_group}: {zone} {model_name} {x_label}->{y_label} (n={len(sub):,})")
                    curves, logs = compute_all_curves_for_relation(
                        x_raw,
                        y_raw,
                        n_bins=n_bins,
                        min_count_per_bin=min_count_per_bin,
                        frac=frac,
                        poly_degree=poly_degree,
                        include_symbolic=include_symbolic,
                        n_grid=n_grid,
                        random_state=random_state,
                    )

                    relation_key = (zone, x_label, y_label, model_name)
                    relation_results[relation_key] = {
                        "analysis_group": analysis_group,
                        "zone": zone,
                        "x_label": x_label,
                        "y_label": y_label,
                        "model": model_name,
                        "x_raw": x_raw,
                        "y_raw": y_raw,
                        "sub": sub,
                        "curves": curves,
                        "n_points": len(sub),
                    }

                    for row in logs:
                        log_records.append({
                            "analysis_group": analysis_group,
                            "zone": zone,
                            "model": model_name,
                            "x_label": x_label,
                            "y_label": y_label,
                            "method": row["method"],
                            "status": row["status"],
                            "reason": row["reason"],
                            "n_points": len(sub),
                        })

                    for method, curve in curves.items():
                        base = {
                            "analysis_group": analysis_group,
                            "zone": zone,
                            "model": model_name,
                            "x_label": x_label,
                            "y_label": y_label,
                            "method": method,
                            "n_raw_points": len(sub),
                        }
                        shape = compute_shape_metrics(curve["x"], curve["y"])
                        strength = compute_strength_metrics(curve["x"], curve["y"], y_raw)
                        tightness = compute_tightness_metrics(x_raw, y_raw, curve["x"], curve["y"])
                        record = {**base, **shape, **strength, **tightness}
                        record["shape_label"] = classify_shape_label(record)
                        if "expression" in curve:
                            record["symbolic_expression"] = curve["expression"]
                        if "breakpoint" in curve:
                            record["piecewise_breakpoint"] = curve["breakpoint"]
                        metric_records.append(record)
                        curve_records.extend(make_curve_records(zone, model_name, x_label, y_label, method, curve))

    metrics_df = pd.DataFrame(metric_records)
    curves_df = pd.DataFrame(curve_records)
    log_df = pd.DataFrame(log_records)
    return relation_results, metrics_df, curves_df, log_df

## 11. Save fitted curves and metrics

Persist the main CSV outputs requested for curves, all metrics, target-specific metrics, and fitting logs.


In [11]:
relation_results, metrics_df, curves_df, fitting_log_df = run_batch_analysis()

curves_df.to_csv(CURVE_DIR / "fitted_curves_all.csv", index=False)
metrics_df.to_csv(METRIC_DIR / "form_candidate_metrics_all.csv", index=False)
metrics_df.query("analysis_group == 'HW_T'").to_csv(METRIC_DIR / "form_candidate_metrics_HW_T.csv", index=False)
metrics_df.query("analysis_group == 'all_zones_E_soil'").to_csv(METRIC_DIR / "form_candidate_metrics_all_zones_E_soil.csv", index=False)
fitting_log_df.to_csv(TABLE_DIR / "fitting_method_log.csv", index=False)

print("Saved curves:", CURVE_DIR / "fitted_curves_all.csv", curves_df.shape)
print("Saved metrics:", METRIC_DIR / "form_candidate_metrics_all.csv", metrics_df.shape)
print("Saved fitting log:", TABLE_DIR / "fitting_method_log.csv", fitting_log_df.shape)
print(fitting_log_df["status"].value_counts(dropna=False).to_dict())


Fitting HW_T: HW CLASSIC P->T (n=12,340)
Fitting HW_T: HW LPJ-GUESS P->T (n=12,538)
Fitting HW_T: HW CLASSIC LAI->T (n=12,340)
Fitting HW_T: HW LPJ-GUESS LAI->T (n=12,538)
Fitting all_zones_E_soil: HW CLASSIC P->E_soil (n=12,340)
Fitting all_zones_E_soil: HW LPJ-GUESS P->E_soil (n=12,538)
Fitting all_zones_E_soil: HW CLASSIC LAI->E_soil (n=12,340)
Fitting all_zones_E_soil: HW LPJ-GUESS LAI->E_soil (n=12,538)
Fitting all_zones_E_soil: HD CLASSIC P->E_soil (n=11,147)
Fitting all_zones_E_soil: HD LPJ-GUESS P->E_soil (n=11,341)
Fitting all_zones_E_soil: HD CLASSIC LAI->E_soil (n=11,147)
Fitting all_zones_E_soil: HD LPJ-GUESS LAI->E_soil (n=11,341)
Fitting all_zones_E_soil: CW CLASSIC P->E_soil (n=21,322)
Fitting all_zones_E_soil: CW LPJ-GUESS P->E_soil (n=21,472)
Fitting all_zones_E_soil: CW CLASSIC LAI->E_soil (n=21,322)
Fitting all_zones_E_soil: CW LPJ-GUESS LAI->E_soil (n=21,472)
Fitting all_zones_E_soil: CD CLASSIC P->E_soil (n=13,445)
Fitting all_zones_E_soil: CD LPJ-GUESS P->E_soil (

## 12. Plotting functions

Create combined response-curve figures, one-method-per-panel figures, and clean summary metric figures.


In [12]:
def sample_scatter(result, sample_size=30000, random_state=42):
    sub = result["sub"]
    if len(sub) > sample_size:
        sub_plot = sub.sample(sample_size, random_state=random_state)
    else:
        sub_plot = sub
    return sub_plot.iloc[:, 0].to_numpy(), sub_plot.iloc[:, 1].to_numpy()


def plot_combined_2x2(relation_results, zone, y_label, save_stem):
    x_labels = ["P", "LAI"]
    model_names = ["CLASSIC", "LPJ-GUESS"]
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.2), dpi=180)

    y_limits_by_row = {}
    for row, x_label in enumerate(x_labels):
        y_arrays = []
        for model_name in model_names:
            key = (zone, x_label, y_label, model_name)
            if key in relation_results and len(relation_results[key]["y_raw"]):
                y_arrays.append(relation_results[key]["y_raw"])
        if y_arrays:
            y_all = np.concatenate(y_arrays)
            ymin, ymax = np.nanpercentile(y_all, [1, 99])
            yrange = ymax - ymin if np.isfinite(ymax - ymin) and (ymax - ymin) > EPS else 1.0
            y_limits_by_row[row] = (ymin - 0.05 * yrange, ymax + 0.05 * yrange)

    for row, x_label in enumerate(x_labels):
        for col, model_name in enumerate(model_names):
            ax = axes[row, col]
            key = (zone, x_label, y_label, model_name)
            if key not in relation_results:
                ax.axis("off")
                continue
            result = relation_results[key]
            x_plot, y_plot = sample_scatter(result, sample_size=30000)
            ax.scatter(x_plot, y_plot, s=3, alpha=0.18, color="0.70", linewidths=0, zorder=1)
            for method in METHOD_ORDER:
                curve = result["curves"].get(method)
                if curve is None:
                    continue
                ax.plot(curve["x"], curve["y"], color=METHOD_COLORS.get(method, "black"), linewidth=1.25, label=method, zorder=5)
            ax.text(
                0.03,
                0.97,
                model_name,
                transform=ax.transAxes,
                fontsize=12,
                fontweight="bold",
                ha="left",
                va="top",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="0.65", alpha=0.90),
                zorder=20,
            )
            ax.set_xlabel(full_label_map.get(x_label, x_label), fontsize=12.5)
            ax.set_ylabel(full_label_map.get(y_label, y_label), fontsize=12.5)
            ax.set_title(f"{short_label_map.get(x_label, x_label)} -> {short_label_map.get(y_label, y_label)}", fontsize=13, pad=9)
            if row in y_limits_by_row:
                ax.set_ylim(y_limits_by_row[row])
            style_axis(ax)

    handles = [Line2D([0], [0], color=METHOD_COLORS[m], lw=1.7, label=m) for m in METHOD_ORDER]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=True, fontsize=10.5, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{zone_labels.get(zone, zone)}: fitted response curves for {short_label_map.get(y_label, y_label)}", fontsize=15, y=0.985)
    fig.tight_layout(rect=[0.03, 0.05, 1, 0.95])
    save_figure(fig, save_stem)


def plot_each_method_panel(relation_results, zone, y_label, save_stem):
    relation_specs = []
    for model_name in ["CLASSIC", "LPJ-GUESS"]:
        for x_label in ["P", "LAI"]:
            relation_specs.append((model_name, x_label))

    fig, axes = plt.subplots(len(relation_specs), len(METHOD_ORDER), figsize=(20, 10.5), dpi=180, squeeze=False)
    for row, (model_name, x_label) in enumerate(relation_specs):
        key = (zone, x_label, y_label, model_name)
        result = relation_results.get(key)
        if result is None:
            continue
        x_plot, y_plot = sample_scatter(result, sample_size=12000)
        for col, method in enumerate(METHOD_ORDER):
            ax = axes[row, col]
            ax.scatter(x_plot, y_plot, s=2, alpha=0.12, color="0.70", linewidths=0, zorder=1)
            curve = result["curves"].get(method)
            if curve is not None:
                ax.plot(curve["x"], curve["y"], color=METHOD_COLORS.get(method, "black"), linewidth=2.0, zorder=5)
            else:
                ax.text(0.5, 0.5, "Unavailable", transform=ax.transAxes, ha="center", va="center", fontsize=10, color="0.35")
            if row == 0:
                ax.set_title(METHOD_SHORT.get(method, method), fontsize=12, pad=8)
            if col == 0:
                ax.set_ylabel(f"{model_name}\n{short_label_map.get(x_label, x_label)} -> {short_label_map.get(y_label, y_label)}", fontsize=12)
            else:
                ax.set_ylabel("")
            if row == len(relation_specs) - 1:
                ax.set_xlabel(full_label_map.get(x_label, x_label), fontsize=11)
            else:
                ax.set_xlabel("")
            style_axis(ax)
    fig.suptitle(f"{zone_labels.get(zone, zone)}: one fitting method per panel for {short_label_map.get(y_label, y_label)}", fontsize=15, y=0.995)
    fig.tight_layout(rect=[0.02, 0.02, 1, 0.965])
    save_figure(fig, save_stem)


def plot_shape_summary(metrics_df):
    shape_counts = (
        metrics_df.groupby(["zone", "model", "y_label", "method", "shape_label"], dropna=False)
        .size()
        .reset_index(name="count")
    )
    shape_counts.to_csv(TABLE_DIR / "shape_label_counts.csv", index=False)

    panel_keys = metrics_df[["zone", "y_label"]].drop_duplicates().sort_values(["y_label", "zone"]).to_records(index=False).tolist()
    models = ["CLASSIC", "LPJ-GUESS"]
    labels = sorted(metrics_df["shape_label"].dropna().unique().tolist())
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(labels), 1)))
    label_colors = dict(zip(labels, colors))

    fig, axes = plt.subplots(len(panel_keys), len(models), figsize=(18, max(10, 2.4 * len(panel_keys))), dpi=180, squeeze=False)
    x = np.arange(len(METHOD_ORDER))
    for row, (zone, y_label) in enumerate(panel_keys):
        for col, model in enumerate(models):
            ax = axes[row, col]
            bottom = np.zeros(len(METHOD_ORDER))
            for label in labels:
                counts = []
                for method in METHOD_ORDER:
                    mask = (
                        (shape_counts["zone"] == zone)
                        & (shape_counts["model"] == model)
                        & (shape_counts["y_label"] == y_label)
                        & (shape_counts["method"] == method)
                        & (shape_counts["shape_label"] == label)
                    )
                    counts.append(int(shape_counts.loc[mask, "count"].sum()))
                ax.bar(x, counts, bottom=bottom, color=label_colors[label], width=0.72, label=label)
                bottom += np.asarray(counts)
            ax.set_title(f"{zone} {short_label_map.get(y_label, y_label)} | {model}", fontsize=11)
            ax.set_xticks(x)
            ax.set_xticklabels([METHOD_SHORT[m] for m in METHOD_ORDER], rotation=35, ha="right", fontsize=8.5)
            ax.set_ylabel("Count")
            style_axis(ax)
    handles = [Line2D([0], [0], color=label_colors[label], lw=8, label=label) for label in labels]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.005), fontsize=9)
    fig.suptitle("Derived shape label counts by zone, output, model, and method", fontsize=15, y=0.997)
    fig.tight_layout(rect=[0.02, 0.05, 1, 0.97])
    save_figure(fig, "shape_label_counts_by_zone_model_output")
    return shape_counts


def plot_metric_dot_summary(metrics_df, metrics, title, save_stem, table_name):
    cols = ["zone", "model", "x_label", "y_label", "method", *metrics]
    summary_df = metrics_df[cols].copy()
    summary_df.to_csv(TABLE_DIR / table_name, index=False)

    fig, axes = plt.subplots(len(metrics), 1, figsize=(15, 3.4 * len(metrics)), dpi=180, squeeze=False)
    method_positions = {method: idx for idx, method in enumerate(METHOD_ORDER)}
    model_offsets = {"CLASSIC": -0.12, "LPJ-GUESS": 0.12}
    marker_map = {"P": "o", "LAI": "s"}
    color_map = {"CLASSIC": "#4C78A8", "LPJ-GUESS": "#F58518"}

    for row, metric in enumerate(metrics):
        ax = axes[row, 0]
        for _, rec in summary_df.iterrows():
            value = rec[metric]
            if not np.isfinite(value):
                continue
            x = method_positions.get(rec["method"], 0) + model_offsets.get(rec["model"], 0)
            # Small deterministic jitter separates zones without changing group identity.
            zone_jitter = {"HW": -0.035, "HD": -0.012, "CW": 0.012, "CD": 0.035}.get(rec["zone"], 0.0)
            ax.scatter(
                x + zone_jitter,
                value,
                s=36,
                color=color_map.get(rec["model"], "0.25"),
                marker=marker_map.get(rec["x_label"], "o"),
                alpha=0.78,
                edgecolor="white",
                linewidth=0.4,
                zorder=5,
            )
        ax.set_ylabel(metric.replace("_", " "))
        ax.set_xticks(range(len(METHOD_ORDER)))
        ax.set_xticklabels([METHOD_SHORT[m] for m in METHOD_ORDER], rotation=25, ha="right")
        style_axis(ax)

    legend_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map["CLASSIC"], markeredgecolor="white", markersize=7, label="CLASSIC"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map["LPJ-GUESS"], markeredgecolor="white", markersize=7, label="LPJ-GUESS"),
        Line2D([0], [0], marker="o", color="0.35", markerfacecolor="0.35", linestyle="None", markersize=6, label="P input"),
        Line2D([0], [0], marker="s", color="0.35", markerfacecolor="0.35", linestyle="None", markersize=6, label="LAI input"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=True, bbox_to_anchor=(0.5, -0.005))
    fig.suptitle(title, fontsize=15, y=0.995)
    fig.tight_layout(rect=[0.03, 0.07, 1, 0.97])
    save_figure(fig, save_stem)
    return summary_df


## 13. Run analysis: HW zone T

Generate the combined 2×2 figure and method-panel figure for warm-wet zone transpiration.


In [13]:
plot_combined_2x2(relation_results, zone="HW", y_label="T", save_stem="HW_T_combined_2x2")
plot_each_method_panel(relation_results, zone="HW", y_label="T", save_stem="HW_T_each_method_4x7")
print("Saved HW T figures")


Saved HW T figures


## 14. Run analysis: all-zone E_soil

Generate the combined 2×2 and method-panel figures for soil evaporation in all four climate zones.


In [14]:
for zone in ["HW", "HD", "CW", "CD"]:
    plot_combined_2x2(relation_results, zone=zone, y_label="E_soil", save_stem=f"{zone}_E_soil_combined_2x2")
    plot_each_method_panel(relation_results, zone=zone, y_label="E_soil", save_stem=f"{zone}_E_soil_each_method_4x7")
print("Saved all-zone E_soil figures")


Saved all-zone E_soil figures


## 15. Summary tables

Save compact summary tables for shape counts and selected strength/tightness metrics.


In [15]:
shape_counts_df = (
    metrics_df.groupby(["zone", "model", "x_label", "y_label", "method", "shape_label"], dropna=False)
    .size()
    .reset_index(name="count")
)
shape_counts_df.to_csv(TABLE_DIR / "shape_label_counts_by_relation.csv", index=False)

strength_metric_columns = [
    "amplitude_norm_by_y_sd",
    "normalized_total_variation",
    "mean_standardized_absolute_slope",
    "late_to_early_absolute_slope_ratio",
]

tightness_metric_columns = [
    "r_squared",
    "nrmse_by_sd",
    "nrmse_by_raw_y_range",
    "mean_buffer_width_by_sd",
    "buffer_width95_by_sd",
]

metrics_df[["analysis_group", "zone", "model", "x_label", "y_label", "method", "shape_label", *strength_metric_columns, *tightness_metric_columns]].to_csv(
    TABLE_DIR / "selected_metric_summary.csv",
    index=False,
)

SELECTED_WIDE_METHODS = [
    "Equal-width bins",
    "Equal-count bins",
    "LOWESS",
    "GAM",
]

WIDE_METHOD_LABELS = {
    "Equal-width bins": "Equal-width bins",
    "Equal-count bins": "Equal-count bins",
    "LOWESS": "LOWESS",
    "GAM": "Generalized Additive Model (GAM)",
}

shape_wide_metrics = [
    "shape_label",
    "pearson_r",
    "spearman_rho",
    "kendall_tau",
    "abs_pearson_r",
    "abs_spearman_r",
    "abs_spearman_minus_abs_pearson",
    "raw_net_change",
    "absolute_net_change",
    "normalized_net_change",
    "normalized_absolute_net_change",
    "positive_slope_fraction",
    "negative_slope_fraction",
    "flat_slope_fraction",
    "dominant_slope_sign",
    "monotonic_fraction",
    "number_of_slope_sign_changes",
    "early_slope",
    "middle_slope",
    "late_slope",
    "early_late_slope_sign_consistency",
    "early_late_slope_sign_opposition",
    "has_internal_valley",
    "has_internal_peak",
    "number_of_internal_extrema",
    "turning_point_x_norm",
    "turning_point_type",
    "two_line_test_evidence",
]

strength_wide_metrics = [
    "raw_amplitude",
    "amplitude_norm_by_y_sd",
    "amplitude_norm_by_y_range",
    "strength_raw_net_change",
    "strength_absolute_net_change",
    "net_change_by_sd",
    "absolute_net_change_by_sd",
    "strength_normalized_net_change",
    "strength_normalized_absolute_net_change",
    "raw_total_variation",
    "total_variation_by_sd",
    "normalized_total_variation",
    "mean_absolute_slope",
    "median_absolute_slope",
    "max_absolute_slope",
    "slope_standard_deviation",
    "slope_interquartile_range",
    "overall_standardized_slope",
    "mean_standardized_absolute_slope",
    "max_standardized_absolute_slope",
    "early_absolute_standardized_slope",
    "middle_absolute_standardized_slope",
    "late_absolute_standardized_slope",
    "middle_to_early_absolute_slope_ratio",
    "late_to_early_absolute_slope_ratio",
    "accumulated_absolute_variation_final_value",
    "accumulated_absolute_variation_early_value",
    "accumulated_absolute_variation_middle_value",
    "accumulated_absolute_variation_late_value",
]

tightness_wide_metrics = [
    "r_squared",
    "rmse",
    "nrmse_by_sd",
    "nrmse_by_raw_y_range",
    "residual_variance",
    "residual_standard_deviation",
    "mean_absolute_residual",
    "median_absolute_residual",
    "residual_iqr",
    "buffer_width_90",
    "buffer_width_95",
    "buffer_width90_by_sd",
    "buffer_width95_by_sd",
    "mean_buffer_width_by_sd",
    "early_rmse",
    "early_rmse_by_sd",
    "early_residual_variance",
    "early_mean_absolute_residual",
    "early_median_absolute_residual",
    "early_buffer_width_90",
    "early_buffer_width_95",
    "early_buffer_width90_by_sd",
    "early_buffer_width95_by_sd",
    "middle_rmse",
    "middle_rmse_by_sd",
    "middle_residual_variance",
    "middle_mean_absolute_residual",
    "middle_median_absolute_residual",
    "middle_buffer_width_90",
    "middle_buffer_width_95",
    "middle_buffer_width90_by_sd",
    "middle_buffer_width95_by_sd",
    "late_rmse",
    "late_rmse_by_sd",
    "late_residual_variance",
    "late_mean_absolute_residual",
    "late_median_absolute_residual",
    "late_buffer_width_90",
    "late_buffer_width_95",
    "late_buffer_width90_by_sd",
    "late_buffer_width95_by_sd",
]


def save_selected_method_wide_table(metrics_df, metric_columns, output_name):
    id_cols = ["analysis_group", "zone", "model", "x_label", "y_label"]
    metric_columns = [c for c in metric_columns if c in metrics_df.columns]
    subset = metrics_df.loc[metrics_df["method"].isin(SELECTED_WIDE_METHODS), id_cols + ["method", *metric_columns]].copy()
    subset["method_display"] = subset["method"].map(WIDE_METHOD_LABELS)
    long_df = subset.melt(
        id_vars=id_cols + ["method_display"],
        value_vars=metric_columns,
        var_name="test",
        value_name="value",
    )
    long_df["wide_column"] = long_df["method_display"] + " | " + long_df["test"]
    wide_df = long_df.pivot(index=id_cols, columns="wide_column", values="value").reset_index()
    wide_df.insert(3, "relation_label", wide_df["model"] + ": " + wide_df["x_label"] + " -> " + wide_df["y_label"])

    ordered_metric_columns = []
    for method in SELECTED_WIDE_METHODS:
        display = WIDE_METHOD_LABELS[method]
        for metric in metric_columns:
            col = f"{display} | {metric}"
            if col in wide_df.columns:
                ordered_metric_columns.append(col)
    wide_df = wide_df[[*id_cols[:3], "relation_label", *id_cols[3:], *ordered_metric_columns]]
    wide_df.to_csv(TABLE_DIR / output_name, index=False)
    return wide_df

shape_wide_df = save_selected_method_wide_table(metrics_df, shape_wide_metrics, "shape_metrics_wide_selected_methods.csv")
strength_wide_df = save_selected_method_wide_table(metrics_df, strength_wide_metrics, "strength_metrics_wide_selected_methods.csv")
tightness_wide_df = save_selected_method_wide_table(metrics_df, tightness_wide_metrics, "tightness_metrics_wide_selected_methods.csv")

print("Saved summary tables")
print("Wide selected-method tables:", shape_wide_df.shape, strength_wide_df.shape, tightness_wide_df.shape)

Saved summary tables
Wide selected-method tables: (20, 118) (20, 110) (20, 170)


## 16. Summary figures

Create PPT-ready summary figures for shape labels, selected strength metrics, and selected tightness metrics.


In [16]:
shape_counts_for_plot = plot_shape_summary(metrics_df)
strength_summary_df = plot_metric_dot_summary(
    metrics_df,
    strength_metric_columns,
    title="Strength candidate metrics",
    save_stem="strength_summary_metrics",
    table_name="strength_summary_metrics.csv",
)
tightness_summary_df = plot_metric_dot_summary(
    metrics_df,
    tightness_metric_columns,
    title="Tightness candidate metrics",
    save_stem="tightness_summary_metrics",
    table_name="tightness_summary_metrics.csv",
)
print("Saved summary figures")


Saved summary figures


## 17. Notes on interpretation

Slope-based Shape and Strength metrics use min-max normalized fitted `x` and fitted `y`, so they are comparable across inputs, models, and climate zones. Strength `normalized_amplitude` is now defined as `raw_amplitude / SD(y_raw)`; `amplitude_norm_by_y_range` is kept separately for range-normalized comparisons. Accumulated response strength uses accumulated absolute normalized variation, `cumsum(abs(diff(y_fit_norm)))`, rather than an unweighted cumulative sum of local slopes. Tightness metrics use raw residuals from `residual = y_raw - interpolated_y_fit`; binned methods are only evaluated over the fitted bin-center range to avoid extrapolation. Overall and local Tightness now include SD-normalized RMSE and buffer-width metrics using `SD(y_raw)` from the full finite raw relation. Formal U-shape tests that are not robustly implemented are kept as `NaN`; the notebook provides geometric turning-point proxies and a two-line slope proxy for interpretation.

In [17]:
required_outputs = [
    METRIC_DIR / "form_candidate_metrics_all.csv",
    METRIC_DIR / "form_candidate_metrics_HW_T.csv",
    METRIC_DIR / "form_candidate_metrics_all_zones_E_soil.csv",
    CURVE_DIR / "fitted_curves_all.csv",
    TABLE_DIR / "fitting_method_log.csv",
    TABLE_DIR / "shape_metrics_wide_selected_methods.csv",
    TABLE_DIR / "strength_metrics_wide_selected_methods.csv",
    TABLE_DIR / "tightness_metrics_wide_selected_methods.csv",
    FIGURE_DIR / "HW_T_combined_2x2.png",
    FIGURE_DIR / "HW_T_each_method_4x7.png",
    FIGURE_DIR / "shape_label_counts_by_zone_model_output.png",
    FIGURE_DIR / "strength_summary_metrics.png",
    FIGURE_DIR / "tightness_summary_metrics.png",
]
required_outputs.extend(FIGURE_DIR / f"{zone}_E_soil_combined_2x2.png" for zone in ["HW", "HD", "CW", "CD"])
required_outputs.extend(FIGURE_DIR / f"{zone}_E_soil_each_method_4x7.png" for zone in ["HW", "HD", "CW", "CD"])

missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    print("Missing required outputs:")
    for item in missing:
        print(" -", item)
else:
    print("All required outputs are present.")

print("Metrics rows:", len(metrics_df))
print("Curve rows:", len(curves_df))
print("Fitting log statuses:", fitting_log_df["status"].value_counts(dropna=False).to_dict())
print("Strength update: normalized_amplitude now equals amplitude_norm_by_y_sd; accumulated_slope columns now store accumulated absolute variation, cumsum(abs(diff(y_fit_norm))).")
print("Tightness update: SD-normalized overall/local RMSE and buffer-width metrics are included.")
print("Wide tables use selected methods: Equal-width bins, Equal-count bins, LOWESS, and Generalized Additive Model (GAM).")

All required outputs are present.
Metrics rows: 140
Curve rows: 291253
Fitting log statuses: {'success': 140}
Strength update: normalized_amplitude now equals amplitude_norm_by_y_sd; accumulated_slope columns now store accumulated absolute variation, cumsum(abs(diff(y_fit_norm))).
Tightness update: SD-normalized overall/local RMSE and buffer-width metrics are included.
Wide tables use selected methods: Equal-width bins, Equal-count bins, LOWESS, and Generalized Additive Model (GAM).
